In [1]:
import math
import random
import copy
import networkx as nx
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import defaultdict


random.seed(42)
np.random.seed(42)



In [2]:
print("\n[1/6] Importazione grafo")
G = nx.read_edgelist("Dataset/facebook_combined.txt")
n_nodes = G.number_of_nodes()
n_edges = G.number_of_edges()
tutti_i_gradi = [grado for nodo, grado in G.degree()]
grado_massimo = max(tutti_i_gradi)
grado_minimo = min(tutti_i_gradi)
print(f"      Nodi: {n_nodes}, Archi: {n_edges}")
print(f"      Grado medio: {2*n_edges/n_nodes:.2f}")
print(f"      Grado massimo: {grado_massimo}")
print(f"      Grado minimo: {grado_minimo}")
print(f"      Clustering medio: {nx.average_clustering(G):.4f}")



[1/6] Importazione grafo
      Nodi: 4039, Archi: 88234
      Grado medio: 43.69
      Grado massimo: 1045
      Grado minimo: 1
      Clustering medio: 0.6055


In [16]:
"""Visualizza proprietà del grafo."""
out_path = "/Users/giuseppepiosorrentino/RetiSocialiProj/Results/ALLGREEDY/"
degrees = [d for _, d in G.degree()]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(degrees, bins=30, color="#2196F3", edgecolor="white", alpha=0.85)
axes[0].set_title("Distribuzione dei gradi", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Grado", fontsize=11)
axes[0].set_ylabel("Frequenza", fontsize=11)
axes[0].grid(True, alpha=0.3)

# Draw small version of the graph
pos = nx.spring_layout(G, seed=42, k=0.5)
nx.draw_networkx(G, pos=pos, ax=axes[1], node_size=20, node_color="#2196F3",
                   edge_color="#BBBBBB", with_labels=False, alpha=0.8, width=0.5)
axes[1].set_title("Visualizzazione della rete", fontsize=13, fontweight="bold")
axes[1].axis("off")

fig.suptitle(f"Rete: n={G.number_of_nodes()}, m={G.number_of_edges()}, "
             f"<k>={2*G.number_of_edges()/G.number_of_nodes():.1f}",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.close()
print(f"  Salvato: {out_path}")

  Salvato: /Users/giuseppepiosorrentino/RetiSocialiProj/Results/ALLGREEDY/


In [4]:
import random
import math

def cost_random(G, low=1, high=5, seed=42):
    rng = random.Random(seed)
    return {v: rng.randint(low, high) for v in G.nodes()}

def cost_half_degree(G):
    return {v: max(1, math.ceil(G.degree(v) / 2)) for v in G.nodes()}

#VALUTIAMO, è OPZIONALE
def cost_custom(G):
    clust = nx.clustering(G)
    return {v: max(1, round(1 + clust[v] * 10)) for v in G.nodes()}

# Istanzia le tre funzioni costo sul grafo G
c_random = cost_random(G)
c_half   = cost_half_degree(G)
#OPZIONALE
c_custom = cost_custom(G)

print("Esempio costi (primi 5 nodi):")
for v in list(G.nodes())[:5]:
    print(f"  nodo {v}: random={c_random[v]}, d/2={c_half[v]}, custom={c_custom[v]}")



Esempio costi (primi 5 nodi):
  nodo 0: random=1, d/2=174, custom=1
  nodo 1: random=1, d/2=9, custom=5
  nodo 2: random=3, d/2=5, custom=10
  nodo 3: random=2, d/2=9, custom=7
  nodo 4: random=2, d/2=5, custom=10


In [5]:
costo_totale_random = sum(c_random.values())
costo_totale_half   = sum(c_half.values())
costo_totale_custom = sum(c_custom.values())
k_random = int(0.10 * costo_totale_random)
k_half   = int(0.10 * costo_totale_half)
k_custom = int(0.10 * costo_totale_custom)

print(f"Costo totale random: {costo_totale_random}")
print(f"Costo totale half:   {costo_totale_half}")
print(f"Costo totale custom: {costo_totale_custom}")

Costo totale random: 12173
Costo totale half:   89243
Costo totale custom: 28467


In [6]:
from functools import lru_cache
import numpy as np

# Pre-calcola strutture del grafo una volta sola
def precompute_graph(G):
    neighbors = {v: set(G.neighbors(v)) for v in G.nodes()}
    degrees   = {v: G.degree(v) for v in G.nodes()}
    thresholds = {v: math.ceil(degrees[v] / 2) for v in G.nodes()}
    return neighbors, degrees, thresholds

neighbors, degrees, thresholds = precompute_graph(G)

def f1(G, S):
    S_set = frozenset(S)
    val = 0
    for v in G.nodes():
        overlap = len(neighbors[v] & S_set)
        val += min(overlap, thresholds[v])
    return val

def f2(G, S):
    S_set = frozenset(S)
    val = 0
    for v in G.nodes():
        overlap = len(neighbors[v] & S_set)
        t = thresholds[v]
        # somma aritmetica invece del ciclo: t + (t-1) + ... 
        if overlap > 0:
            effective = min(overlap, t)
            val += effective * t - (effective * (effective - 1)) // 2
    return val

def f3(G, S):
    S_set = frozenset(S)
    val = 0
    for v in G.nodes():
        dv = degrees[v]
        if dv == 0:
            continue
        overlap = len(neighbors[v] & S_set)
        t = thresholds[v]
        for i in range(1, overlap + 1):
            denom = dv - i + 1
            if denom > 0:
                val += max((t - i + 1) / denom, 0)
    return val

In [7]:
def majority_cascade(G, S):
    """
    Simula il processo di attivazione con regola Majority.
    Un nodo v si attiva se almeno metà dei suoi vicini sono già attivi.
    Restituisce Inf[S] = insieme di tutti i nodi attivati.
    """
    infected = set(S)
    
    while True:
        new_infected = set()
        for v in G.nodes():
            if v not in infected:
                dv = G.degree(v)
                if dv == 0:
                    continue
                neighbors_infected = sum(1 for u in G.neighbors(v) if u in infected)
                if neighbors_infected >= math.ceil(dv / 2):
                    new_infected.add(v)
        
        if not new_infected:  # nessun nuovo nodo attivato: la cascata si ferma
            break
        infected |= new_infected
    
    return infected

In [8]:
from functools import lru_cache

def marginal_gain(G, v, S_current, fi_func):
    """
    Ottimizzazione: invece di ricalcolare fi su tutto il grafo,
    guarda solo i vicini di v che cambiano.
    """
    S_set = frozenset(S_current)
    val = 0
    # Solo i vicini di v sono influenzati dall'aggiunta di v
    for u in neighbors[v]:
        overlap_before = len(neighbors[u] & S_set)
        overlap_after  = overlap_before + (1 if v not in S_set else 0)
        
        if fi_func == f1:
            val += min(overlap_after, thresholds[u]) - min(overlap_before, thresholds[u])
        elif fi_func == f2:
            t = thresholds[u]
            def contrib(ov):
                effective = min(ov, t)
                return effective * t - (effective * (effective - 1)) // 2
            val += contrib(overlap_after) - contrib(overlap_before)
        elif fi_func == f3:
            dv = degrees[u]
            if dv == 0:
                continue
            t = thresholds[u]
            before = sum(max((t-i+1)/( dv-i+1), 0) for i in range(1, overlap_before+1))
            after  = sum(max((t-i+1)/(dv-i+1), 0)  for i in range(1, overlap_after+1))
            val += after - before
    return val


In [9]:
def cost_seeds_greedy(G, k, c, fi_func, verbose=False):
    S_p = set()
    S_d = set()
    iteration = 0

    # Cache dei ratio: ricalcola solo i nodi il cui vicinato è cambiato
    ratio_cache = {v: fi_func(G, {v}) / c[v] for v in G.nodes() if c[v] > 0}

    while True:
        iteration += 1
        if not ratio_cache:
            break

        best_u = max(ratio_cache, key=ratio_cache.get)
        S_p = set(S_d)
        S_d = S_d | {best_u}
        del ratio_cache[best_u]

        if sum(c[u] for u in S_d) > k:
            break

        # Aggiorna solo i nodi vicini di best_u
        for v in neighbors[best_u]:
            if v not in S_d and c.get(v, 0) > 0:
                ratio_cache[v] = marginal_gain(G, v, S_d, fi_func) / c[v]

        if verbose:
            print(f"  iter {iteration}: nodo {best_u}, costo={sum(c[u] for u in S_d)}/{k}")

    return S_p

In [10]:
S_greedy_f1 = cost_seeds_greedy(G, k_random, c_random, f1)
print(f"Seed set Greedy-f1: {len(S_greedy_f1)} nodi, costo={sum(c_random[u] for u in S_greedy_f1)}")
print(f"\n|Inf[G, S_f1]| = {len(majority_cascade(G, S_greedy_f1))}")

Seed set Greedy-f1: 750 nodi, costo=1217

|Inf[G, S_f1]| = 2826


In [11]:
S_greedy_f2 = cost_seeds_greedy(G, k_random, c_random, f2)
print(f"Seed set Greedy-f2: {len(S_greedy_f2)} nodi, costo={sum(c_random[u] for u in S_greedy_f2)}")
print(f"|Inf[G, S_f2]| = {len(majority_cascade(G, S_greedy_f2))}")

Seed set Greedy-f2: 856 nodi, costo=1217
|Inf[G, S_f2]| = 2803


In [13]:
S_greedy_f3 = cost_seeds_greedy(G, k_random, c_random, f3)
print(f"Seed set Greedy-f3: {len(S_greedy_f3)} nodi, costo={sum(c_random[u] for u in S_greedy_f3)}")
print(f"|Inf[G, S_f3]| = {len(majority_cascade(G, S_greedy_f3))}")

Seed set Greedy-f3: 823 nodi, costo=1217
|Inf[G, S_f3]| = 3323


In [14]:
# Aggiungi punti intermedi nel range interessante
percentages = [0.05, 0.06, 0.07, 0.08, 0.09, 
               0.10, 0.12, 0.15, 0.20, 0.25, 
               0.30, 0.40, 0.50]
budgets_random = [int(p * sum(c_random.values())) for p in percentages]

inf_greedy_f1 = []
inf_greedy_f2 = []
inf_greedy_f3 = []

for k in budgets_random:
    S_f1 = cost_seeds_greedy(G, k, c_random, f1)
    S_f2 = cost_seeds_greedy(G, k, c_random, f2)
    S_f3 = cost_seeds_greedy(G, k, c_random, f3)
    
    inf_greedy_f1.append(len(majority_cascade(G, S_f1)))
    inf_greedy_f2.append(len(majority_cascade(G, S_f2)))
    inf_greedy_f3.append(len(majority_cascade(G, S_f3)))
    
    print(f"k={k:5d} | f1={inf_greedy_f1[-1]:4d} | f2={inf_greedy_f2[-1]:4d} | f3={inf_greedy_f3[-1]:4d}")

k=  608 | f1=1052 | f2= 661 | f3= 677
k=  730 | f1=1321 | f2=1216 | f3= 777
k=  852 | f1=1390 | f2=1867 | f3=1740
k=  973 | f1=2195 | f2=2119 | f3=2356
k= 1095 | f1=2628 | f2=2258 | f3=3103
k= 1217 | f1=2826 | f2=2803 | f3=3323
k= 1460 | f1=3195 | f2=3186 | f3=3674
k= 1825 | f1=3614 | f2=3439 | f3=3833
k= 2434 | f1=3882 | f2=3818 | f3=3993
k= 3043 | f1=3990 | f2=3906 | f3=4026
k= 3651 | f1=4032 | f2=3944 | f3=4039
k= 4869 | f1=4039 | f2=4023 | f3=4039
k= 6086 | f1=4039 | f2=4039 | f3=4039


In [ ]:
plt.figure(figsize=(9, 5))

plt.plot(percentages, inf_greedy_f1, color="#2196F3", marker="o", linewidth=2, label="Greedy-f1")
plt.plot(percentages, inf_greedy_f2, color="#E91E63", marker="s", linewidth=2, label="Greedy-f2")
plt.plot(percentages, inf_greedy_f3, color="#4CAF50", marker="^", linewidth=2, label="Greedy-f3")

plt.axhline(y=G.number_of_nodes(), color="gray", linestyle="--", alpha=0.5, label=f"|V|={G.number_of_nodes()}")
plt.xlabel("Percentuale del costo totale")
plt.ylabel("|Inf[G, S]|")
plt.title("Greedy — |Inf[G,S]| al variare di k (c_random)") 
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(out_path + "greedy_random", dpi=150, bbox_inches="tight")
plt.show()

/var/folders/8z/67y4_fv93g3dgx30jl7w8ss40000gn/T/ipykernel_2189/497407484.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
# Aggiungi punti intermedi nel range interessante
percentages = [0.05, 0.06, 0.07, 0.08, 0.09, 
               0.10, 0.12, 0.15, 0.20, 0.25, 
               0.30, 0.40, 0.50]
budgets_random = [int(p * sum(c_half.values())) for p in percentages]

inf_greedy_f1 = []
inf_greedy_f2 = []
inf_greedy_f3 = []

for k in budgets_random:
    S_f1 = cost_seeds_greedy(G, k, c_half, f1)
    S_f2 = cost_seeds_greedy(G, k, c_half, f2)
    S_f3 = cost_seeds_greedy(G, k, c_half, f3)
    
    inf_greedy_f1.append(len(majority_cascade(G, S_f1)))
    inf_greedy_f2.append(len(majority_cascade(G, S_f2)))
    inf_greedy_f3.append(len(majority_cascade(G, S_f3)))
    
    print(f"k={k:5d} | f1={inf_greedy_f1[-1]:4d} | f2={inf_greedy_f2[-1]:4d} | f3={inf_greedy_f3[-1]:4d}")

In [ ]:
plt.figure(figsize=(9, 5))

plt.plot(percentages, inf_greedy_f1, color="#2196F3", marker="o", linewidth=2, label="Greedy-f1")
plt.plot(percentages, inf_greedy_f2, color="#E91E63", marker="s", linewidth=2, label="Greedy-f2")
plt.plot(percentages, inf_greedy_f3, color="#4CAF50", marker="^", linewidth=2, label="Greedy-f3")

plt.axhline(y=G.number_of_nodes(), color="gray", linestyle="--", alpha=0.5, label=f"|V|={G.number_of_nodes()}")
plt.xlabel("Percentuale del costo totale")
plt.ylabel("|Inf[G, S]|")
plt.title("Greedy — |Inf[G,S]| al variare di k (c_half)") 
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(out_path + "greedy_half", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Aggiungi punti intermedi nel range interessante
percentages = [0.05, 0.06, 0.07, 0.08, 0.09, 
               0.10, 0.12, 0.15, 0.20, 0.25, 
               0.30, 0.40, 0.50]
budgets_random = [int(p * sum(c_custom.values())) for p in percentages]

inf_greedy_f1 = []
inf_greedy_f2 = []
inf_greedy_f3 = []

for k in budgets_random:
    S_f1 = cost_seeds_greedy(G, k, c_custom, f1)
    S_f2 = cost_seeds_greedy(G, k, c_custom, f2)
    S_f3 = cost_seeds_greedy(G, k, c_custom, f3)
    
    inf_greedy_f1.append(len(majority_cascade(G, S_f1)))
    inf_greedy_f2.append(len(majority_cascade(G, S_f2)))
    inf_greedy_f3.append(len(majority_cascade(G, S_f3)))
    
    print(f"k={k:5d} | f1={inf_greedy_f1[-1]:4d} | f2={inf_greedy_f2[-1]:4d} | f3={inf_greedy_f3[-1]:4d}")

In [ ]:
plt.figure(figsize=(9, 5))

plt.plot(percentages, inf_greedy_f1, color="#2196F3", marker="o", linewidth=2, label="Greedy-f1")
plt.plot(percentages, inf_greedy_f2, color="#E91E63", marker="s", linewidth=2, label="Greedy-f2")
plt.plot(percentages, inf_greedy_f3, color="#4CAF50", marker="^", linewidth=2, label="Greedy-f3")

plt.axhline(y=G.number_of_nodes(), color="gray", linestyle="--", alpha=0.5, label=f"|V|={G.number_of_nodes()}")
plt.xlabel("Percentuale del costo totale")
plt.ylabel("|Inf[G, S]|")
plt.title("Greedy — |Inf[G,S]| al variare di k (c_custom)") 
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(out_path + "greedy_custom", dpi=150, bbox_inches="tight")
plt.show()